In [1]:
from asyncio import start_server
from calendar import weekday
from select import KQ_NOTE_WRITE

import seaborn as sns
import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import altair as alt
import plotly.express as px
import plotly.graph_objects as go
import great_tables as tg
import datetime as dt
import numpy as np
import pandas as pd


In [2]:
df_path = r'/Users/zygimantas/Downloads/archive (1)/fitness_dataset.csv'

In [3]:
df = pl.read_csv(df_path)

In [4]:
df

age,height_cm,weight_kg,heart_rate,blood_pressure,sleep_hours,nutrition_quality,activity_index,smokes,gender,is_fit
i64,i64,i64,f64,f64,f64,f64,f64,str,str,i64
56,152,65,69.6,117.0,null,2.37,3.97,"""no""","""F""",1
69,186,95,60.8,114.8,7.5,8.77,3.19,"""0""","""F""",1
46,192,103,61.4,116.4,null,8.2,2.03,"""0""","""F""",0
32,189,83,60.2,130.1,7.0,6.18,3.68,"""0""","""M""",1
60,175,99,58.1,115.8,8.0,9.95,4.83,"""yes""","""F""",1
…,…,…,…,…,…,…,…,…,…,…
52,173,98,60.7,106.1,null,1.54,3.25,"""1""","""M""",1
61,186,74,51.4,123.8,9.4,8.63,3.15,"""no""","""M""",1
77,198,89,76.7,103.6,8.3,1.98,3.36,"""yes""","""M""",0


In [5]:
df = df.with_columns(
    cs.integer().shrink_dtype()
)

In [6]:
df.null_count()

age,height_cm,weight_kg,heart_rate,blood_pressure,sleep_hours,nutrition_quality,activity_index,smokes,gender,is_fit
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,160,0,0,0,0,0


In [7]:
df = df.with_columns(
    pl.col('sleep_hours').fill_null(0.0)
)

In [8]:
df.describe()

statistic,age,height_cm,weight_kg,heart_rate,blood_pressure,sleep_hours,nutrition_quality,activity_index,smokes,gender,is_fit
str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64
"""count""",2000.0,2000.0,2000.0,2000.0,2000.0,2000.0,2000.0,2000.0,"""2000""","""2000""",2000.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0""","""0""",0.0
"""mean""",49.114,174.533,83.5405,70.2886,119.90885,6.91225,5.03514,2.99904,null,null,0.3995
"""std""",17.926564,14.37175,25.852534,11.846339,14.578032,2.49646,2.864156,1.136383,null,null,0.489918
"""min""",18.0,150.0,30.0,45.0,90.0,0.0,0.0,1.0,"""0""","""F""",0.0
"""25%""",34.0,162.0,64.0,62.1,109.7,6.1,2.55,2.04,null,null,0.0
"""50%""",49.0,174.0,83.0,70.3,120.0,7.4,5.07,2.98,null,null,0.0
"""75%""",65.0,187.0,102.0,78.4,129.8,8.4,7.47,3.95,null,null,1.0
"""max""",79.0,199.0,250.0,118.6,171.2,12.0,10.0,4.99,"""yes""","""M""",1.0


In [9]:
X = df.select(
    cs.exclude('is_fit')
)

In [10]:
y = df.get_column(
    'is_fit'
)

In [11]:
X_numeric = df.select(
    cs.numeric()
)

X_categorical = df.select(
    cs.string()
)

In [12]:
X_categorical.select(
    pl.all().n_unique()
)

smokes,gender
u32,u32
4,2


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [14]:
column_transformer = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(
            handle_unknown='ignore', sparse_output=False, drop='first'),
        X.select(cs.string()).columns),
        ('passthrough', 'passthrough', X.select(cs.numeric()).columns)
    ]
)

In [15]:
column_transformer

,transformers,"[('encoder', ...), ('passthrough', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,False


In [16]:
X_encoded = column_transformer.fit_transform(X)

In [17]:
feature_names = column_transformer.named_transformers_['encoder'].get_feature_names_out()

In [18]:
numeric_columns = X.select(cs.numeric()).columns

In [19]:
all_feature_names = list(feature_names) + list(numeric_columns)

In [20]:
all_feature_names

['smokes_1',
 'smokes_no',
 'smokes_yes',
 'gender_M',
 'age',
 'height_cm',
 'weight_kg',
 'heart_rate',
 'blood_pressure',
 'sleep_hours',
 'nutrition_quality',
 'activity_index']

In [21]:
X_encoded = pl.DataFrame(
    X_encoded, schema=all_feature_names
)

In [22]:
from sklearn.model_selection import train_test_split

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

In [24]:
from sklearn.preprocessing import StandardScaler

In [25]:
scaler = StandardScaler()

In [26]:
scaler

,copy,True
,with_mean,True
,with_std,True


In [27]:
scaled_X_train = scaler.fit_transform(X_train)

In [28]:
scaled_X_test = scaler.transform(X_test)

In [29]:
scaled_X_train = pl.DataFrame(scaled_X_train, schema=X_train.columns)

In [30]:
scaled_X_test = pl.DataFrame(scaled_X_test, schema=X_test.columns)

In [32]:
pandas_df = df.to_pandas()

fig = px.histogram(pandas_df, x='is_fit', title='Distribution of is_fit', color='is_fit')
fig.show()


In [35]:
fig = px.histogram(pandas_df, x='is_fit', title='Distribution of is_fit', color='is_fit')
fig.show()

In [37]:
numerical_cols = df.select(cs.numeric().exclude('is_fit')).columns
for col in numerical_cols:
    fig = px.box(pandas_df, x='is_fit', y=col, title=f'Distribution of {col} by Fitness Level', color='is_fit')
    fig.show()


In [39]:
from sklearn.linear_model import LinearRegression

In [42]:
liner_model = LinearRegression(n_jobs=-1)

In [43]:
liner_model.fit(scaled_X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,-1
,positive,False


In [44]:
predictions = liner_model.predict(X_test)

In [45]:
predictions

array([ -4.81280832,  -4.00579917,  -3.39591619,  -6.11168294,
        -7.02263272,  -7.42419073,  -2.26885083,  -7.54483527,
        -1.84934272,  -3.0709085 ,  -6.92333381,  -3.90020375,
        -4.4553089 ,  -3.95944336,  -7.87538046,  -1.79204242,
        -3.87670633,  -6.2254521 ,  -6.14540645,  -7.1613819 ,
        -1.08051359,  -6.10486967,  -2.79400515,  -8.97959568,
        -7.55155483,  -3.28769137,  -6.61579983,  -3.33578108,
        -5.61237781,  -3.03656137,  -0.58269226,  -3.26007123,
        -1.80741541,  -7.6080458 ,  -6.6478141 ,  -4.92605836,
        -3.47566445,  -6.67272537,  -0.98067515,  -4.62294453,
        -7.98209391,  -3.133931  ,  -2.91930814,  -6.26836535,
        -8.30749299,  -3.37801033,  -2.96326837,  -1.55139298,
        -7.3019033 ,  -2.14797645,  -5.47496425,  -6.03278254,
        -1.48602688,  -6.68448997,  -6.79082636,  -8.34894163,
        -2.65300767,  -7.306009  ,  -7.45112373,  -6.32587637,
        -7.82929179,  -4.21113474,  -8.03166228,  -3.93

In [46]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

In [49]:
pl.DataFrame({
    'actual': y_test,
    'predicted': predictions,
    'error': y_test - predictions,
    'error_percent': (y_test - predictions) / y_test * 100,
    'error_abs_percent': np.abs(y_test - predictions) / y_test * 100,
    'error_squared': (y_test - predictions) ** 2,
})

actual,predicted,error,error_percent,error_abs_percent,error_squared
i8,f64,f64,f64,f64,f64
1,-4.812808,5.812808,581.280832,581.280832,33.788741
1,-4.005799,5.005799,500.579917,500.579917,25.058025
1,-3.395916,4.395916,439.591619,439.591619,19.324079
1,-6.111683,7.111683,711.168294,711.168294,50.576034
0,-7.022633,7.022633,inf,inf,49.31737
…,…,…,…,…,…
0,-3.694622,3.694622,inf,inf,13.650233
0,-8.151689,8.151689,inf,inf,66.45004
0,-3.933982,3.933982,inf,inf,15.476212


In [54]:
MSE = mean_squared_error(y_test, predictions)
RMSE = np.sqrt(MSE)
MAE = mean_absolute_error(y_test, predictions)
MAE = mean_absolute_percentage_error(y_test, predictions)
R2 = r2_score(y_test, predictions)

In [55]:
f'MSE {MSE:.2f}, RMSE {RMSE:.2f}, MAE {MAE:.2f}'

'MSE 29.85, RMSE 5.46, MAE 14184009746659248.00'

In [67]:
results = pl.DataFrame({
    'Error': [MSE, RMSE, MAE, MAE, R2],
    'Metric': ['MSE', 'RMSE', 'MAE', 'MAPE', 'R2']
})

In [57]:
from sklearn.model_selection import GridSearchCV

In [58]:
liner_reg_parm_grid = {
    'fit_intercept': [True, False],
    'positive': [True, False],
    'copy_X': [True, False],
    'n_jobs': [-1]
}

In [59]:
grid_search = GridSearchCV(
    estimator=LinearRegression(),
    param_grid=liner_reg_parm_grid,
    cv=5,
    verbose=1,
    n_jobs=-1
)

In [60]:
grid_search.fit(scaled_X_train, y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


,estimator,LinearRegression()
,param_grid,"{'copy_X': [True, False], 'fit_intercept': [True, False], 'n_jobs': [-1], 'positive': [True, False]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,fit_intercept,True


In [61]:
predictions = grid_search.predict(X_test)

In [62]:
pl.DataFrame({
    'actual': y_test,
    'predicted': grid_search.predict(X_test),
    'error': (grid_search.predict(X_test) - y_test),
    'error_percent': (grid_search.predict(X_test) - y_test) / y_test,
})

actual,predicted,error,error_percent
i8,f64,f64,f64
1,-4.812808,-5.812808,-5.812808
1,-4.005799,-5.005799,-5.005799
1,-3.395916,-4.395916,-4.395916
1,-6.111683,-7.111683,-7.111683
0,-7.022633,-7.022633,-inf
…,…,…,…
0,-3.694622,-3.694622,-inf
0,-8.151689,-8.151689,-inf
0,-3.933982,-3.933982,-inf


In [63]:
MSE = mean_squared_error(y_test, grid_search.predict(X_test))
RMSE = np.sqrt(MSE)
MAE = mean_absolute_error(y_test, grid_search.predict(X_test))
r2 = r2_score(y_test, grid_search.predict(X_test))
percent_error = mean_absolute_percentage_error(y_test, grid_search.predict(X_test))

In [64]:
liner_reg_cv_values = [MSE, RMSE, r2, MAE, percent_error]

In [68]:
results = results.with_columns(pl.Series(
    name='Linera Regression CV', values=liner_reg_cv_values
))

In [69]:
results

Error,Metric,Linera Regression CV
f64,str,f64
29.852826,"""MSE""",29.852826
5.463774,"""RMSE""",5.463774
5.118945,"""MAE""",-122.183808
5.118945,"""MAPE""",5.118945
-122.183808,"""R2""",1.4184e16


In [71]:
from sklearn.svm import SVR

In [72]:
svm = SVR()

In [99]:
svm_cv_parms = {
    'kernel': ['poly', 'linear', 'rbf', 'sigmoid'],
    'degree': [1, 2, 3, 4],
    'gamma': ['auto'],
    'epsilon': [0.01, 0.1, 0.5, 1]
}

In [100]:
svr_grid_search = GridSearchCV(
    estimator=svm,
    param_grid=svm_cv_parms,
    cv=5,
    verbose=1,
    n_jobs=-1
)

In [101]:
svr_grid_search.fit(scaled_X_train, y_train)

Fitting 7 folds for each of 64 candidates, totalling 448 fits


,estimator,SVR()
,param_grid,"{'degree': [1, 2, ...], 'epsilon': [0.01, 0.1, ...], 'gamma': ['auto'], 'kernel': ['poly', 'linear', ...]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,7
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,kernel,'rbf'


In [102]:
svr_grid_search.predict(X_test)

array([0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572847,
       0.24572847, 0.24572847, 0.24572847, 0.24572847, 0.24572

In [103]:
svr_grid_search.best_params_

{'degree': 1, 'epsilon': 0.01, 'gamma': 'auto', 'kernel': 'rbf'}

In [104]:
MSE = mean_squared_error(y_test, svr_grid_search.predict(X_test))
RMSE = np.sqrt(MSE)
MAE = mean_absolute_error(y_test, svr_grid_search.predict(X_test))
r2 = r2_score(y_test, svr_grid_search.predict(X_test))
percent_error = mean_absolute_percentage_error(y_test, svr_grid_search.predict(X_test))

In [105]:
result = results.with_columns(
    pl.Series(name='SVR CV', values=[MSE, RMSE, r2, MAE, percent_error])
)

In [106]:
print(result)

shape: (5, 4)
┌─────────────┬────────┬──────────────────────┬───────────┐
│ Error       ┆ Metric ┆ Linera Regression CV ┆ SVR CV    │
│ ---         ┆ ---    ┆ ---                  ┆ ---       │
│ f64         ┆ str    ┆ f64                  ┆ f64       │
╞═════════════╪════════╪══════════════════════╪═══════════╡
│ 29.852826   ┆ MSE    ┆ 29.852826            ┆ 0.270156  │
│ 5.463774    ┆ RMSE   ┆ 5.463774             ┆ 0.519766  │
│ 5.118945    ┆ MAE    ┆ -122.183808          ┆ -0.114766 │
│ 5.118945    ┆ MAPE   ┆ 5.118945             ┆ 0.455502  │
│ -122.183808 ┆ R2     ┆ 1.4184e16            ┆ 6.5016e14 │
└─────────────┴────────┴──────────────────────┴───────────┘


In [107]:
from sklearn.neighbors import KNeighborsClassifier

In [108]:
knn_cv_parms = {
    'n_neighbors': [1, 2, 3, 4 ,5],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'leaf_size': [30, 40, 50],
    'p': [1, 2],
    'metric': ['minkowski'],
    'n_jobs': [-1]
}

In [109]:
knn_grid_search = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=knn_cv_parms,
    cv=5,
    verbose=1,
    n_jobs=-1
)

In [110]:
knn_grid_search.fit(scaled_X_train, y_train)

Fitting 5 folds for each of 240 candidates, totalling 1200 fits


,estimator,KNeighborsClassifier()
,param_grid,"{'algorithm': ['auto', 'ball_tree', ...], 'leaf_size': [30, 40, ...], 'metric': ['minkowski'], 'n_jobs': [-1], ...}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,5


In [112]:
knn_grid_search.predict(X_test)

array([1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
       1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,

In [113]:
knn_grid_search.best_params_

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'n_jobs': -1,
 'n_neighbors': 5,
 'p': 1,
 'weights': 'uniform'}

In [114]:
MSE = mean_squared_error(y_test, knn_grid_search.predict(X_test))
RMSE = np.sqrt(MSE)
MAE = mean_absolute_error(y_test, knn_grid_search.predict(X_test))
r2 = r2_score(y_test, knn_grid_search.predict(X_test))
percent_error = mean_absolute_percentage_error(y_test, knn_grid_search.predict(X_test))

In [115]:
knn_grid_values = [MSE, RMSE, r2, MAE, percent_error]

In [117]:
results = result.with_columns(
    pl.Series(name='KNN CV', values=knn_grid_values)
)

In [118]:
from sklearn.tree import DecisionTreeClassifier

In [125]:
desition_tree_params = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': [1, 2, 3, 4, 5],
    'min_samples_split': [2, 3, 4, 5],
    'min_samples_leaf': [1, 2, 3, 4, 5],
    'min_weight_fraction_leaf': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
    'max_features': [None, 'sqrt', 'log2'],
    'random_state': [42]
}

In [126]:
desition_tree_grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(),
    param_grid=desition_tree_params,
    cv=5,
    verbose=1,
    n_jobs=-1
)

In [127]:
desition_tree_grid_search.fit(scaled_X_train, y_train)

Fitting 5 folds for each of 7200 candidates, totalling 36000 fits


,estimator,DecisionTreeClassifier()
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_depth': [1, 2, ...], 'max_features': [None, 'sqrt', ...], 'min_samples_leaf': [1, 2, ...], ...}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [128]:
desition_tree_grid_search.best_params_

{'criterion': 'gini',
 'max_depth': 5,
 'max_features': None,
 'min_samples_leaf': 3,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'random_state': 42,
 'splitter': 'best'}

In [129]:
predictions = desition_tree_grid_search.predict(X_test)

In [131]:
MSE = mean_squared_error(y_test, predictions)
RMSE = np.sqrt(MSE)
MAE = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
percent_error = mean_absolute_percentage_error(y_test, predictions)

In [132]:
desition_tree_grid_search_values = [MSE, RMSE, r2, MAE, percent_error]

In [134]:
results = results.with_columns(
    pl.Series(name='Desition Tree CV', values=desition_tree_grid_search_values)
)

In [135]:
from sklearn.ensemble import RandomForestClassifier

In [158]:
rfc_cv_params = {
    'n_estimators': [25, 100],
    'max_depth': [3, 5, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True],
    'random_state': [42],
    'n_jobs': [-1]
}

In [159]:
rfc_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=rfc_cv_params,
    cv=5,
    verbose=1,
)

In [160]:
rfc_grid_search.fit(scaled_X_train, y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


,estimator,RandomForestClassifier()
,param_grid,"{'bootstrap': [True], 'max_depth': [3, 5, ...], 'max_features': ['sqrt', 'log2', ...], 'min_samples_leaf': [1, 2], ...}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [161]:
rfc_grid_search.best_params_

{'bootstrap': True,
 'max_depth': None,
 'max_features': 'sqrt',
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 100,
 'n_jobs': -1,
 'random_state': 42}

In [162]:
predictions = rfc_grid_search.predict(X_test)

In [163]:
MSE = mean_squared_error(y_test, predictions)
RMSE = np.sqrt(MSE)
MAE = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
percent_error = mean_absolute_percentage_error(y_test, predictions)

In [165]:
results = results.with_columns(
    pl.Series(name='Random Forest CV', values=[MSE, RMSE, r2, MAE, percent_error])
)

In [166]:
from sklearn.naive_bayes import GaussianNB

In [174]:
gaussian_nb_cv_parms = {
    'var_smoothing': [1e-09, 1e-10, 1e-11],
    'priors': [None],
}

In [175]:
gaussian_nb_grid_search = GridSearchCV(
    estimator=GaussianNB(),
    param_grid=gaussian_nb_cv_parms,
    cv=5,
    verbose=1,
)

In [176]:
gaussian_nb_grid_search.fit(scaled_X_train, y_train)

Fitting 5 folds for each of 3 candidates, totalling 15 fits


,estimator,GaussianNB()
,param_grid,"{'priors': [None], 'var_smoothing': [1e-09, 1e-10, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,priors,None


In [182]:
predictions = gaussian_nb_grid_search.predict(X_test)
predictions

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [179]:
MSE = mean_squared_error(y_test, predictions)
RMSE = np.sqrt(MSE)
MAE = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
percent_error = mean_absolute_percentage_error(y_test, predictions)

In [180]:
results = results.with_columns(
    pl.Series(name='Gaussian NB CV', values=[MSE, RMSE, r2, MAE, percent_error])
)

In [181]:
results

Error,Metric,Linera Regression CV,SVR CV,KNN CV,Desition Tree CV,Random Forest CV,Gaussian NB CV
f64,str,f64,f64,f64,f64,f64,f64
29.852826,"""MSE""",29.852826,0.270156,0.5775,0.57,0.4925,0.4125
5.463774,"""RMSE""",5.463774,0.519766,0.759934,0.754983,0.701783,0.642262
5.118945,"""MAE""",-122.183808,-0.114766,-1.382979,-1.352031,-1.032237,-0.702128
5.118945,"""MAPE""",5.118945,0.455502,0.5775,0.57,0.4925,0.4125
-122.183808,"""R2""",1.4184e16,6.5016e14,2.4545e15,2.5671e15,2.0266e15,0.4125


In [183]:
from sklearn.ensemble import GradientBoostingClassifier

Looking at the selected cell, I need to remove all the comments from the `gbc_cv_params` dictionary definition.



In [192]:
gbc_cv_params = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, None],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
    'loss': ['log_loss'],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 4],
    'max_features': ['sqrt'],
    'criterion': ['friedman_mse']
}

In [193]:
gbc_grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(),
    param_grid=gbc_cv_params,
    cv=5,
    verbose=1,
)

In [194]:
gbc_grid_search.fit(scaled_X_train, y_train)

Fitting 5 folds for each of 96 candidates, totalling 480 fits


,estimator,GradientBoostingClassifier()
,param_grid,"{'criterion': ['friedman_mse'], 'learning_rate': [0.01, 0.1], 'loss': ['log_loss'], 'max_depth': [3, 5, ...], ...}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [195]:
predictions = gbc_grid_search.predict(X_test)

In [196]:
predictions

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [197]:
MSE = mean_squared_error(y_test, predictions)
RMSE = np.sqrt(MSE)
MAE = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
percent_error = mean_absolute_percentage_error(y_test, predictions)

In [199]:
results = results.with_columns(
    pl.Series(name='Gradient Boosting CV', values=[MSE, RMSE, r2, MAE, percent_error])
)

In [202]:
results

Error,Metric,Linera Regression CV,SVR CV,KNN CV,Desition Tree CV,Random Forest CV,Gaussian NB CV,Gradient Boosting CV
f64,str,f64,f64,f64,f64,f64,f64,f64
29.852826,"""MSE""",29.852826,0.270156,0.5775,0.57,0.4925,0.4125,0.565
5.463774,"""RMSE""",5.463774,0.519766,0.759934,0.754983,0.701783,0.642262,0.751665
5.118945,"""MAE""",-122.183808,-0.114766,-1.382979,-1.352031,-1.032237,-0.702128,-1.331399
5.118945,"""MAPE""",5.118945,0.455502,0.5775,0.57,0.4925,0.4125,0.565
-122.183808,"""R2""",1.4184e16,6.5016e14,2.4545e15,2.5671e15,2.0266e15,0.4125,2.5445e15
